# Sequential BIO Training

Trains the **Sequential BIO** system: Phase 1 (cls-dominant, same as System F) →
Phase 2 (BIO span head initialized from Phase 1 encoder).

Completes the 2×2 matrix:

| | Single-pass | Sequential |
|---|---|---|
| QA formulation | E (joint) | F/A/D |
| BIO formulation | E4 ≈ E | **This notebook** |

**Run order:** Cell 1 (Drive setup) → Cell 2 (repo + deps) → Cell 3 (training loop).

**Seeds:** 42, 123, 7 — all three run sequentially in Cell 3.

In [ ]:
# ── Cell 1: Drive mount + durable symlinks ─────────────────────────────────
# MUST run before any training cell. Outputs go to Drive, not ephemeral /content.

import os, shutil, sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Idiomator_Research')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/IdiomBERT')

drive_models  = DRIVE_ROOT / 'models'
drive_results = DRIVE_ROOT / 'results'
drive_models.mkdir(parents=True, exist_ok=True)
drive_results.mkdir(parents=True, exist_ok=True)

def setup_symlinks(repo_dir):
    """Replace real models/ and results/ dirs with Drive symlinks."""
    for name, drive_path in [('models', drive_models), ('results', drive_results)]:
        local = repo_dir / name
        if local.is_symlink():
            assert 'drive' in os.readlink(local).lower(), \
                f"{local} symlink does not point at Drive: {os.readlink(local)}"
            print(f"  ✓ {name}/ already → Drive")
        else:
            if local.exists():
                shutil.rmtree(local)   # remove committed real dir — must go first
            os.symlink(drive_path, local)
            print(f"  ✓ {name}/ → {drive_path}")

print(f"Drive root: {DRIVE_ROOT}")

In [ ]:
# ── Cell 2: Clone repo + install deps ──────────────────────────────────────
# Push your latest code to GitHub before running this cell.

import subprocess

GITHUB_REPO = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH      = 'main'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, GITHUB_REPO, str(REPO_DIR)],
        check=True
    )
else:
    print("Repo already cloned — pulling latest...")
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

# Wire up Drive symlinks now that repo exists
setup_symlinks(REPO_DIR)

# Verify symlinks point at Drive
for p in ('models', 'results'):
    assert os.path.islink(p), f"{p} is a real dir — abort"
    assert 'drive' in os.readlink(p).lower(), f"{p} symlink not at Drive"
    print(f"  ✓ {p} → {os.readlink(p)}")

# Install deps
!pip install -q transformers==4.40.0 scikit-learn tqdm

In [ ]:
# ── Cell 3: Training — all 3 seeds ────────────────────────────────────────
# Runs Phase 1 then Phase 2 BIO for seeds 42, 123, 7.
# Checkpoints saved to Drive after each phase/epoch — session-reset safe.

import json, time
from pathlib import Path

os.chdir(REPO_DIR)

SEEDS = [42, 123, 7]
OUTPUT_BASE = 'models/sequential_bio'
DATA_DIR    = 'data/idioms_structured/Splits'

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — abort'}")
assert torch.cuda.is_available(), "Connect to a GPU runtime first"

summary = {}

for seed in SEEDS:
    seed_dir = Path(OUTPUT_BASE) / f's{seed}'
    p1_metrics_path = seed_dir / 'phase1' / 'metrics.json'
    p2_metrics_path = seed_dir / 'phase2_bio' / 'metrics.json'

    print(f"\n{'='*60}")
    print(f"SEED {seed}")
    print(f"{'='*60}")

    # Phase 1
    if p1_metrics_path.exists():
        print(f"  ✓ Phase 1 already done (seed {seed}) — skipping")
    else:
        t0 = time.time()
        ret = subprocess.run(
            ['python', '-u', 'training/Train_Sequential_BIO.py',
             '--seed',       str(seed),
             '--output_dir', OUTPUT_BASE,
             '--data_dir',   DATA_DIR,
             '--phase',      '1'],
            check=True
        )
        print(f"  Phase 1 done in {(time.time()-t0)/60:.1f} min")

    # Phase 2 BIO
    if p2_metrics_path.exists():
        print(f"  ✓ Phase 2 BIO already done (seed {seed}) — skipping")
    else:
        t0 = time.time()
        subprocess.run(
            ['python', '-u', 'training/Train_Sequential_BIO.py',
             '--seed',       str(seed),
             '--output_dir', OUTPUT_BASE,
             '--data_dir',   DATA_DIR,
             '--phase',      '2'],
            check=True
        )
        print(f"  Phase 2 BIO done in {(time.time()-t0)/60:.1f} min")

    # Read back from Drive to confirm persistence
    assert p1_metrics_path.exists(), f"Phase 1 metrics missing at {p1_metrics_path}"
    assert p2_metrics_path.exists(), f"Phase 2 BIO metrics missing at {p2_metrics_path}"
    p1m = json.loads(p1_metrics_path.read_text())
    p2m = json.loads(p2_metrics_path.read_text())
    summary[seed] = {
        'p1_test_cls_f1':      p1m['test_cls_macro_f1'],
        'p2_test_span_exact':  p2m['test_span_exact'],
        'p2_test_span_overlap': p2m['test_span_overlap'],
    }
    print(f"  Seed {seed} — cls F1: {p1m['test_cls_macro_f1']:.4f} | "
          f"span exact: {p2m['test_span_exact']:.4f} | "
          f"overlap F1: {p2m['test_span_overlap']:.4f}")

print("\n" + "="*60)
print("ALL SEEDS DONE")
print("="*60)
print(f"{'Seed':<8} {'Cls F1':<10} {'Span Exact':<12} {'Overlap F1'}")
for seed, m in summary.items():
    print(f"{seed:<8} {m['p1_test_cls_f1']:<10.4f} "
          f"{m['p2_test_span_exact']:<12.4f} {m['p2_test_span_overlap']:.4f}")

In [ ]:
# ── Cell 4: Joint evaluation (Phase 1 cls × Phase 2 BIO span) ─────────────
# Compute joint F1 inline using the same logic as Full_evaluation.py.
# This gives the equivalent of System F's joint metric but with BIO span.

import json
from pathlib import Path
from collections import defaultdict

def load_preds_by_id(path):
    preds = {}
    for line in open(path, encoding='utf-8'):
        r = json.loads(line)
        key = (r['language'], r['sentence'], r['span_start'], r['span_end'])
        preds[key] = r
    return preds

def overlap_f1(pred_s, pred_e, gold_s, gold_e):
    if pred_s is None or pred_e is None:
        return 0.0
    ps = set(range(pred_s, pred_e))
    gs = set(range(gold_s, gold_e))
    if not ps or not gs:
        return 0.0
    ov = len(ps & gs)
    if ov == 0:
        return 0.0
    p = ov / len(ps); r = ov / len(gs)
    return 2 * p * r / (p + r)

def compute_joint_f1(p1_preds, p2_preds):
    """
    Joint F1: correct = cls correct AND span overlap F1 > 0.
    Only idiomatic examples contribute to span score.
    """
    lang_scores = defaultdict(list)
    for key, s1 in p1_preds.items():
        if key not in p2_preds:
            continue
        s2 = p2_preds[key]
        gold_cls  = s1['idiomaticity']
        pred_cls  = s1.get('pred_idiomaticity', 'idiomatic')
        lang      = s1['language']

        cls_correct = (pred_cls == gold_cls)
        if gold_cls == 'idiomatic':
            ov = overlap_f1(
                s2.get('pred_span_start'), s2.get('pred_span_end'),
                s1['span_start'], s1['span_end']
            ) if cls_correct else 0.0
            joint = float(cls_correct) * ov
        else:
            joint = float(cls_correct)
        lang_scores[lang].append(joint)

    import numpy as np
    all_scores = [s for v in lang_scores.values() for s in v]
    return {
        'overall': float(np.mean(all_scores)),
        'per_lang': {lang: float(np.mean(v)) for lang, v in lang_scores.items()},
        'n': len(all_scores),
    }

print(f"{'Seed':<6} {'Joint F1':<12} {'EN':<8} {'ES':<8} {'HI':<8} {'TE':<8}")
joint_scores = []
for seed in SEEDS:
    seed_dir = Path(OUTPUT_BASE) / f's{seed}'
    p1_path  = seed_dir / 'phase1' / 'test_predictions.jsonl'
    p2_path  = seed_dir / 'phase2_bio' / 'test_predictions.jsonl'

    if not p1_path.exists() or not p2_path.exists():
        print(f"{seed:<6} MISSING PREDS")
        continue

    p1 = load_preds_by_id(p1_path)
    p2 = load_preds_by_id(p2_path)
    jf = compute_joint_f1(p1, p2)
    joint_scores.append(jf['overall'])

    langs_order = ['English', 'Spanish', 'Hindi', 'Telugu']
    per = jf['per_lang']
    print(f"{seed:<6} {jf['overall']:<12.4f} "
          f"{per.get('English',0):<8.4f} {per.get('Spanish',0):<8.4f} "
          f"{per.get('Hindi',0):<8.4f} {per.get('Telugu',0):<8.4f}")

if joint_scores:
    import numpy as np
    print(f"\nMean Joint F1: {np.mean(joint_scores):.4f} ± {np.std(joint_scores):.4f}")
    print("\nCompare to:")
    print("  System F  (seq QA):    ~0.7XXX  ← check key_numbers.md")
    print("  System E  (joint QA):  ~0.7XXX  ← check key_numbers.md")
    print("  System E4 (joint BIO): ~0.7XXX  ← check key_numbers.md")

In [ ]:
# ── Cell 5: Save aggregate results JSON to Drive ───────────────────────────
import json, time
from pathlib import Path

agg = {
    'system': 'sequential_bio',
    'description': 'Phase1 cls-dominant (same as F) + Phase2 BIO span head from Phase1 encoder',
    'timestamp': time.strftime('%Y-%m-%d'),
    'seeds': {str(seed): summary.get(seed, {}) for seed in SEEDS},
    'joint_f1_seeds': joint_scores,
}
if joint_scores:
    import numpy as np
    agg['joint_f1_mean'] = float(np.mean(joint_scores))
    agg['joint_f1_std']  = float(np.std(joint_scores))

out = Path(OUTPUT_BASE) / 'aggregate_results.json'
out.write_text(json.dumps(agg, indent=2))
print(f"Saved → {out}")
print(json.dumps(agg, indent=2))